# Chapter 9: Learning and Adaptation

EliteAI Academy

Explore reward-driven adaptation, few-shot prompting, online learning, memory, self-improvement, evolutionary search, and label generation.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, random, copy
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 max_tokens: int = 512) -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=max_tokens, system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 max_tokens: int = 512) -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: LEARNING READY"))

## 1. The Problem — Static Agents Degrade in Dynamic Environments

In [ ]:
STATIC_SYSTEM = "You are a customer support agent. Be helpful."

test_queries = [
    "My order hasn't arrived after 3 weeks.",
    "Your product broke after one day. I'm furious.",
    "Can I return something I bought 6 months ago?",
]

print("=== STATIC AGENT (no learning) ===")
for q in test_queries:
    r = call_llm(q, system=STATIC_SYSTEM)
    print(f"\n❓ {q}")
    print(f"🤖 {r[:200]}")

print("\n⚠️  Same generic system prompt regardless of feedback.")
print("   No improvement. No personalisation. No adaptation.")

## 2. Learning Paradigms Overview

In [ ]:
overview = """
╔══════════════════════╦══════════════════════════════════════╦═══════════════════════╗
║ Paradigm             ║ How It Works                         ║ Agent Use Case        ║
╠══════════════════════╬══════════════════════════════════════╬═══════════════════════╣
║ Reinforcement        ║ Actions → rewards/penalties →        ║ Game AI, robot        ║
║ Learning (RL)        ║ policy update                        ║ control, trading bots ║
╠══════════════════════╬══════════════════════════════════════╬═══════════════════════╣
║ Supervised           ║ Labelled examples → model trains     ║ Email classifier,     ║
║ Learning             ║ to map inputs → outputs              ║ fraud detection       ║
╠══════════════════════╬══════════════════════════════════════╬═══════════════════════╣
║ Unsupervised         ║ Unlabelled data → discover patterns  ║ Customer segmentation ║
║ Learning             ║ and clusters                         ║ anomaly detection     ║
╠══════════════════════╬══════════════════════════════════════╬═══════════════════════╣
║ Few-Shot / Zero-Shot ║ LLM adapts from examples or          ║ New task adaptation   ║
║ (In-Context)         ║ instructions in the prompt           ║ rapid prototyping     ║
╠══════════════════════╬══════════════════════════════════════╬═══════════════════════╣
║ Online Learning      ║ Model updates continuously with      ║ Real-time recommender ║
║                      ║ each new data point                  ║ fraud detection       ║
╠══════════════════════╬══════════════════════════════════════╬═══════════════════════╣
║ Memory-Based         ║ Recall past episodes →               ║ Personalised agents   ║
║ Learning             ║ apply to current decisions           ║ support bots          ║
╚══════════════════════╩══════════════════════════════════════╩═══════════════════════╝
"""
print(overview)

## 3. Paradigm 1 — Reward-Driven Adaptation (Simulated RL)

The agent tries a response, receives a reward signal, and updates its strategy to maximise future rewards.

In [ ]:
@dataclass
class Experience:
    'One RL experience tuple: (state, action, reward, next_state)'
    state:      str
    action:     str
    reward:     float
    feedback:   str
    timestamp:  str = field(default_factory=lambda: datetime.now().isoformat()[:19])

class RLAgent:
    'LLM agent that learns from reward signals.'
    def __init__(self, initial_policy: str):
        self.policy         = initial_policy
        self.experience_buffer: List[Experience] = []
        self.reward_history: List[float] = []
        self.version        = 1

    def act(self, state: str) -> str:
        'Generate a response using the current policy.'
        return call_llm(state, system=self.policy)

    def store_experience(self, state: str, action: str, reward: float, feedback: str):
        'Store experience in the replay buffer.'
        self.experience_buffer.append(Experience(state, action, reward, feedback))
        self.reward_history.append(reward)

    def update_policy(self, min_experiences: int = 2):
        'Update the policy (system prompt) based on accumulated experiences.'
        if len(self.experience_buffer) < min_experiences:
            return

        exp_text = "\n".join(
            f"State: {e.state[:80]}\n"
            f"Action: {e.action[:100]}\n"
            f"Reward: {e.reward:+.1f} | Feedback: {e.feedback}\n"
            for e in self.experience_buffer[-5:]
        )

        new_policy = call_llm(
            f"""Current agent policy:
{self.policy}

Recent experiences (state → action → reward → feedback):
{exp_text}

Rewrite the policy to improve future performance.
Focus on: reinforcing what received positive rewards,
avoiding what received negative rewards.
Return ONLY the new policy text.""",
            system="You are a policy optimiser. Improve agent instructions based on reward signals."
        )
        self.policy = new_policy
        self.version += 1
        avg_reward = sum(self.reward_history[-5:]) / min(5, len(self.reward_history))
        print(f"\n  📈 Policy updated → v{self.version} | Avg reward (last 5): {avg_reward:+.2f}")

    def performance_chart(self):
        print("\n📊 Reward history:")
        for i, r in enumerate(self.reward_history):
            bar = "█" * max(0, int((r + 1) * 5))
            print(f"  Step {i+1:02d}: [{bar:<10}] {r:+.1f}")

rl_agent = RLAgent(
    initial_policy="You are a customer support agent. Be helpful and polite."
)

episodes = [
    ("My order is 3 weeks late!",           -0.8, "Response was too vague. Should offer concrete steps and apology."),
    ("Your product broke immediately!",      -0.6, "No empathy shown. Should acknowledge frustration first."),
    ("Can I exchange my item?",              +0.7, "Good — provided clear policy. Could add a direct link."),
    ("I've been waiting 30 mins on hold!",   -0.5, "Should acknowledge wait time frustration immediately."),
    ("The replacement you sent is perfect!", +0.9, "Excellent — warm, personalised, asks for further help."),
]

print("=== REINFORCEMENT LEARNING AGENT ===")
for i, (state, reward, feedback) in enumerate(episodes):
    print(f"\n[Episode {i+1}]")
    action = rl_agent.act(state)
    print(f"  State:  {state}")
    print(f"  Action: {action[:150]}")
    print(f"  Reward: {reward:+.1f} | Feedback: {feedback}")
    rl_agent.store_experience(state, action, reward, feedback)

    if (i + 1) % 3 == 0:
        rl_agent.update_policy()

print(f"\n{'='*55}\nFINAL POLICY (v{rl_agent.version}):\n{rl_agent.policy}")
rl_agent.performance_chart()

## 4. PPO vs DPO — Concept Comparison

In [ ]:
comparison = """
╔══════════════════════════════════╦══════════════════════════════════════════════╗
║  PPO (Proximal Policy Optimiz.)  ║  DPO (Direct Preference Optimiz.)            ║
╠══════════════════════════════════╬══════════════════════════════════════════════╣
║  Two-step process:               ║  Single-step process:                        ║
║  1. Train a reward model         ║  Uses preference data DIRECTLY               ║
║     from human preference data   ║  to update the LLM policy                    ║
║  2. Fine-tune LLM with PPO       ║                                              ║
║     to maximise reward model     ║                                              ║
╠══════════════════════════════════╬══════════════════════════════════════════════╣
║  Risk: LLM may "hack" the reward ║  No reward model = no hacking risk           ║
║  model (reward hacking)          ║                                              ║
╠══════════════════════════════════╬══════════════════════════════════════════════╣
║  Key mechanism: clipped          ║  Key mechanism: directly increases prob.     ║
║  objective — "trust region"      ║  of preferred responses, decreases           ║
║  prevents too-large updates      ║  prob. of rejected responses                 ║
╠══════════════════════════════════╬══════════════════════════════════════════════╣
║  Complex, more moving parts      ║  Simpler, more stable                        ║
║  Better for sequential decisions ║  Better for LLM alignment                    ║
╠══════════════════════════════════╬══════════════════════════════════════════════╣
║  Use: Robot control, game AI,    ║  Use: Chatbot alignment, preference          ║
║  complex agent policies          ║  learning, RLHF replacement                  ║
╚══════════════════════════════════╩══════════════════════════════════════════════╝

IN-CONTEXT ANALOGY:
  PPO = hire a referee (reward model) → coach trains based on referee's scores
  DPO = show examples directly → coach adjusts immediately from the examples
"""
print(comparison)

## 5. Paradigm 2 — Few-Shot In-Context Learning

LLMs adapt to new tasks instantly from examples provided in the prompt. No fine-tuning required — the model "learns" within the context window.

In [ ]:
print("=== ZERO-SHOT (no examples) ===")
zero_shot_response = call_llm(
    "Classify this text as POSITIVE, NEGATIVE, or NEUTRAL: 'The battery life is disappointing but the camera is excellent.'",
    system="You are a sentiment classifier."
)
print(f"Result: {zero_shot_response}")

print("\n=== ONE-SHOT (one example) ===")
one_shot_response = call_llm(
    """Classify sentiment. Format: LABEL | REASON

Example:
Text: 'Fast shipping but product was damaged.'
Output: MIXED | Positive shipping experience, negative product condition

Now classify:
Text: 'The battery life is disappointing but the camera is excellent.'
Output:""",
    system="You are a sentiment classifier."
)
print(f"Result: {one_shot_response}")

print("\n=== FEW-SHOT (three examples) ===")

few_shot_examples = [
    ("The product stopped working after 2 days.",          "NEGATIVE | Product failure after very short use"),
    ("Incredible build quality, exactly as described.",   "POSITIVE | Quality and accuracy confirmed"),
    ("Delivery was fast but packaging was damaged.",       "MIXED | Good logistics, poor packaging"),
]

examples_text = "\n\n".join(
    f"Text: '{text}'\nOutput: {label}" for text, label in few_shot_examples
)

few_shot_response = call_llm(
    f"""Classify sentiment. Format: LABEL | REASON\n\n{examples_text}\n\nNow classify:\nText: 'The battery life is disappointing but the camera is excellent.'\nOutput:""",
    system="You are a precise sentiment classifier. Follow the exact format of the examples."
)
print(f"Result: {few_shot_response}")

print("\n=== DYNAMIC FEW-SHOT (adaptive example selection) ===")

example_library = [
    {"text": "App crashes on startup.",            "label": "NEGATIVE | Critical bug",   "domain": "software"},
    {"text": "UI is intuitive and clean.",          "label": "POSITIVE | Good UX design", "domain": "software"},
    {"text": "Subscription price is too high.",     "label": "NEGATIVE | Pricing concern", "domain": "software"},
    {"text": "Fast delivery, product undamaged.",   "label": "POSITIVE | Good logistics",  "domain": "ecommerce"},
    {"text": "Item not as described in listing.",   "label": "NEGATIVE | Misrepresentation","domain": "ecommerce"},
]

def select_examples(query: str, library: list, top_k: int = 2) -> list:
    'Select most relevant examples using LLM scoring.'
    library_text = "\n".join(f"{i}. {e['text']} ({e['domain']})" for i, e in enumerate(library))
    scores_raw = call_llm(
        f"""Query: '{query}'\nWhich {top_k} examples are most similar? Return a JSON array of indices only.
\n{library_text}""",
        system="You are an example selector. Return only valid JSON array of integers."
    )
    try:
        indices = json.loads(re.sub(r"```[\w]*", "", scores_raw).strip())
        return [library[i] for i in indices if i < len(library)]
    except:
        return library[:top_k]

new_query = "The API documentation is outdated and confusing."
selected  = select_examples(new_query, example_library, top_k=2)
dynamic_examples = "\n\n".join(f"Text: '{e['text']}'\nOutput: {e['label']}" for e in selected)

dynamic_result = call_llm(
    f"""Classify sentiment. Format: LABEL | REASON\n\n{dynamic_examples}\n\nNow classify:\nText: '{new_query}'\nOutput:"""
)
print(f"Selected examples from: {[e['domain'] for e in selected]}")
print(f"Result: {dynamic_result}")

## 6. Paradigm 3 — Online Learning

The agent updates its knowledge base continuously with every new interaction. No batch training — real-time adaptation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
import numpy as np

class OnlineLearningAgent:
    'Agent that continuously updates a lightweight classifier'
    def __init__(self):
        self.vectorizer   = TfidfVectorizer(max_features=500)
        self.classifier   = SGDClassifier(loss="hinge", random_state=42)
        self.classes      = ["URGENT", "NORMAL", "LOW"]
        self.is_fitted    = False
        self.n_trained    = 0
        self.seen_texts   = []
        self.seen_labels  = []

    def _preprocess(self, texts: List[str]) -> np.ndarray:
        if not self.is_fitted:
            return self.vectorizer.fit_transform(texts)
        return self.vectorizer.transform(texts)

    def learn(self, text: str, label: str):
        'Update the model with a single new labelled example.'
        self.seen_texts.append(text)
        self.seen_labels.append(label)

        if len(set(self.seen_labels)) < 2:
            return

        if not self.is_fitted:
            X = self.vectorizer.fit_transform(self.seen_texts)
            self.classifier.partial_fit(X, self.seen_labels, classes=self.classes)
            self.is_fitted = True
        else:
            X = self.vectorizer.transform([text])
            self.classifier.partial_fit(X, [label])

        self.n_trained += 1

    def predict(self, text: str) -> dict:
        'Predict priority. Falls back to LLM if classifier not ready.'
        if not self.is_fitted or self.n_trained < 3:

            label = call_llm(
                f"Classify ticket priority as URGENT, NORMAL, or LOW: {text}",
                system="Return only the label word."
            ).strip().upper().split()[0]
            return {"label": label, "source": "llm", "confidence": None}
        else:
            X     = self.vectorizer.transform([text])
            label = self.classifier.predict(X)[0]
            proba = self.classifier.decision_function(X)[0]
            conf  = float(max(proba) - min(proba))
            return {"label": label, "source": "classifier", "confidence": round(conf, 2)}

print("=== ONLINE LEARNING AGENT ===")
ol_agent = OnlineLearningAgent()

ticket_stream = [
    ("Production server is completely down!",           "URGENT"),
    ("Can you add dark mode to the app?",               "LOW"),
    ("Password reset email not arriving.",              "NORMAL"),
    ("Database corrupted — all data may be lost!",      "URGENT"),
    ("Update the FAQ page.",                            "LOW"),
    ("Login page is slow to load.",                     "NORMAL"),
    ("Security breach detected in payment system!",     "URGENT"),
    ("Would like a bulk discount for enterprise plan.", "LOW"),
]

test_ticket = "API is returning 500 errors for all requests."
print(f"\nTest ticket: '{test_ticket}'")

for i, (text, label) in enumerate(ticket_stream):
    ol_agent.learn(text, label)
    pred = ol_agent.predict(test_ticket)
    print(f"  After learning {i+1:02d} examples | Prediction: {pred['label']} "
          f"(source={pred['source']}, confidence={pred['confidence']})")

print(f"\n✅ Final prediction: {ol_agent.predict(test_ticket)}")
print(f"   Total examples learned: {ol_agent.n_trained}")

## 7. Paradigm 4 — Memory-Based Learning

The agent recalls past episodes to inform current decisions. "I solved a similar problem before — here's what worked."

In [ ]:
@dataclass
class Episode:
    problem:     str
    solution:    str
    outcome:     str
    rating:      float
    tags:        List[str] = field(default_factory=list)
    created_at:  str = field(default_factory=lambda: datetime.now().isoformat()[:19])

class MemoryBasedLearningAgent:
    'Agent with episodic memory.'
    def __init__(self, role: str):
        self.role    = role
        self.memory: List[Episode] = []

    def remember(self, problem: str, solution: str, outcome: str, rating: float, tags: list):
        'Store a new episode in episodic memory.'
        ep = Episode(problem, solution, outcome, rating, tags)
        self.memory.append(ep)
        print(f"  💾 Stored episode #{len(self.memory)}: {problem[:50]}... [rating={rating:.1f}]")

    def recall(self, current_problem: str, top_k: int = 3) -> List[Episode]:
        'Retrieve the most relevant past episodes.'
        if not self.memory:
            return []

        episodes_text = "\n".join(
            f"{i}. [{e.outcome.upper()}|{e.rating:.1f}] {e.problem[:80]}"
            for i, e in enumerate(self.memory)
        )
        raw = call_llm(
            f"""Current problem: '{current_problem}'
Which {top_k} past episodes are most relevant? Return a JSON array of indices.
\n{episodes_text}""",
            system="You are a memory retrieval system. Return only valid JSON array of integers."
        )
        try:
            indices = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            return [self.memory[i] for i in indices if i < len(self.memory)]
        except:

            return sorted(self.memory, key=lambda e: e.rating, reverse=True)[:top_k]

    def solve(self, problem: str) -> str:
        'Solve a problem using recalled episodes as context.'
        recalled = self.recall(problem)

        if recalled:
            print(f"  🧠 Recalled {len(recalled)} relevant episodes")
            episodes_context = "\n\n".join(
                f"PAST PROBLEM: {e.problem}\nSOLUTION: {e.solution}\nOUTCOME: {e.outcome} (rating: {e.rating:.1f})"
                for e in recalled if e.outcome != "failure"
            )
            system = f"""You are a {self.role}.
Use these past successful experiences as reference:

{episodes_context}

Apply relevant patterns to solve the new problem."""
        else:
            system = f"You are a {self.role}. Solve this problem carefully."

        return call_llm(problem, system=system)

agent = MemoryBasedLearningAgent("senior data engineer specialising in Azure")

print("=== LOADING EPISODIC MEMORY ===")
agent.remember(
    "ADF pipeline fails with column mismatch on copy activity",
    "Used schema mapping in sink settings to align column names explicitly",
    "success", 0.9, ["ADF", "schema", "copy activity"]
)
agent.remember(
    "MERGE statement fails on REPLICATE distribution tables in Synapse",
    "Switched to truncate-staging + upsert pattern instead of MERGE",
    "success", 0.95, ["Synapse", "MERGE", "distribution"]
)
agent.remember(
    "Databricks notebook failing on Delta schema evolution",
    "Added mergeSchema=True option to the write configuration",
    "success", 0.85, ["Databricks", "Delta", "schema evolution"]
)
agent.remember(
    "ADF pipeline throwing NVARCHAR/CCI incompatibility error",
    "Cast columns to VARCHAR before loading into columnstore index tables",
    "success", 0.88, ["ADF", "Synapse", "CCI", "NVARCHAR"]
)

print("\n=== SOLVING NEW PROBLEM WITH EPISODIC RECALL ===")
new_problem = "My ADF pipeline fails when copying data to a Synapse table with columnstore index. The error mentions data type incompatibility."
print(f"\n❓ Problem: {new_problem}")
solution = agent.solve(new_problem)
print(f"\n🤖 Solution:\n{solution}")

agent.remember(new_problem, solution, "pending", 0.0, ["ADF", "Synapse", "CCI"])

## 8. The Self-Improvement Loop — SICA Pattern

The agent evaluates its own performance, identifies weaknesses, and rewrites its own instructions to improve future performance.

In [ ]:
@dataclass
class AgentVersion:
    version:     int
    instructions: str
    scores:      List[float] = field(default_factory=list)
    notes:       str = ""

    @property
    def avg_score(self) -> float:
        return sum(self.scores) / len(self.scores) if self.scores else 0.0

class SelfImprovingAgent:
    'Implements the SICA self-improvement pattern.'
    def __init__(self, initial_instructions: str, task_domain: str):
        self.task_domain = task_domain
        self.archive: List[AgentVersion] = [
            AgentVersion(version=1, instructions=initial_instructions)
        ]
        self.overseer_log: List[str] = []

    @property
    def current(self) -> AgentVersion:
        'The best-performing version in the archive.'
        return max(self.archive, key=lambda v: v.avg_score)

    def execute(self, task: str) -> str:
        'Run the current best version on a task.'
        return call_llm(task, system=self.current.instructions)

    def evaluate(self, task: str, output: str) -> Tuple[float, str]:
        'Evaluate output quality. Returns (score 0-1, critique).'
        eval_raw = call_llm(
            f"""Evaluate this output for the task domain: {self.task_domain}

Task: {task}
Output: {output}

Return ONLY a JSON object: {{"score": <0.0-1.0>, "critique": "<specific issues>"}}""",
            system="You are an objective evaluator. Return only valid JSON."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", eval_raw).strip())
            return float(data.get("score", 0.5)), data.get("critique", "")
        except:
            return 0.5, "Could not parse evaluation"

    def self_modify(self, recent_critiques: List[str]) -> str:
        'Core SICA mechanism: agent rewrites its own instructions.'
        archive_summary = "\n".join(
            f"v{v.version}: avg_score={v.avg_score:.2f} | {v.notes[:60]}"
            for v in self.archive
        )
        critiques_text = "\n".join(f"- {c}" for c in recent_critiques)

        new_instructions = call_llm(
            f"""You are a {self.task_domain} AI agent improving your own instructions.

Current instructions:
{self.current.instructions}

Version archive (performance history):
{archive_summary}

Recent critiques:
{critiques_text}

Rewrite your instructions to address all critiques and improve performance.
Return ONLY the improved instructions text.""",
            system="You are a self-improving agent. Rewrite instructions to address weaknesses."
        )
        return new_instructions

    def overseer_check(self, iterations_since_improvement: int) -> Optional[str]:
        'Asynchronous overseer monitors for stagnation or loops.'
        if iterations_since_improvement >= 2:
            msg = f"⚠️  OVERSEER: No improvement in {iterations_since_improvement} iterations. Consider broader changes."
            self.overseer_log.append(msg)
            return msg
        return None

    def run_improvement_cycle(self, benchmark_tasks: List[str], max_iterations: int = 3):
        'Run the full SICA self-improvement loop.'
        print(f"\n{'='*60}")
        print(f"SELF-IMPROVEMENT LOOP: {self.task_domain}")
        print(f"{'='*60}")

        no_improve_streak = 0
        best_score_ever   = 0.0

        for iteration in range(max_iterations):
            print(f"\n[ITERATION {iteration+1}/{max_iterations}]")

            overseer_msg = self.overseer_check(no_improve_streak)
            if overseer_msg:
                print(f"  {overseer_msg}")

            critiques = []
            for task in benchmark_tasks:
                output   = self.execute(task)
                score, critique = self.evaluate(task, output)
                self.current.scores.append(score)
                critiques.append(critique)
                print(f"  Task: {task[:50]}... | Score: {score:.2f}")

            avg = self.current.avg_score
            print(f"  → Average score: {avg:.3f}")

            if avg > best_score_ever:
                best_score_ever   = avg
                no_improve_streak = 0
            else:
                no_improve_streak += 1

            print(f"  → Self-modifying instructions...")
            new_instructions = self.self_modify(critiques)
            new_version = AgentVersion(
                version=len(self.archive) + 1,
                instructions=new_instructions,
                notes=f"Iteration {iteration+1} improvement"
            )
            self.archive.append(new_version)
            print(f"  → New version v{new_version.version} created")

        print(f"\n{'='*60}\nSELF-IMPROVEMENT COMPLETE")
        print(f"Best version: v{self.current.version} | Score: {self.current.avg_score:.3f}")
        print(f"Versions created: {len(self.archive)}")
        if self.overseer_log:
            print(f"Overseer interventions: {len(self.overseer_log)}")

INITIAL_INSTRUCTIONS = """You are a Python code generator.
Write working Python code for user requests."""

BENCHMARK_TASKS = [
    "Write a function that validates an email address.",
    "Write a function that calculates the Fibonacci sequence up to n terms.",
    "Write a function to flatten a nested list.",
]

sica = SelfImprovingAgent(
    initial_instructions=INITIAL_INSTRUCTIONS,
    task_domain="Python code generation"
)

sica.run_improvement_cycle(BENCHMARK_TASKS, max_iterations=3)

print(f"\n{'='*60}\nFINAL INSTRUCTIONS (v{sica.current.version})\n{'='*60}")
print(sica.current.instructions)

## 9. Evolutionary Optimisation — AlphaEvolve / OpenEvolve Pattern

A population of solutions evolves over generations. LLM generates mutations; an evaluator scores them; best survive.

In [ ]:
@dataclass
class Candidate:
    code:       str
    score:      float = 0.0
    generation: int   = 0
    parent_id:  Optional[int] = None
    candidate_id: int = field(default_factory=lambda: random.randint(1000, 9999))

class EvolutionaryOptimiser:
    'Evolutionary code optimisation (AlphaEvolve/OpenEvolve pattern).'
    def __init__(self, task: str, evaluator_fn, population_size: int = 3):
        self.task            = task
        self.evaluator       = evaluator_fn
        self.population_size = population_size
        self.population:     List[Candidate] = []
        self.best_ever:      Optional[Candidate] = None
        self.generation      = 0
        self.history:        List[Dict] = []

    def _generate_initial(self) -> List[Candidate]:
        'Generate initial population with different approaches.'
        approaches = ["iterative approach", "recursive approach", "functional approach"]
        candidates = []
        for approach in approaches[:self.population_size]:
            code = call_llm(
                f"Write a Python solution using a {approach} for: {self.task}\nReturn only the code.",
                system="You are an expert Python developer. Write clean, correct code."
            )
            candidates.append(Candidate(code=code, generation=0))
        return candidates

    def _mutate(self, parent: Candidate, critique: str) -> Candidate:
        'LLM-driven mutation: improve the parent based on critique.'
        mutated_code = call_llm(
            f"""Task: {self.task}

Current solution (score: {parent.score:.2f}):
{parent.code}

Critique: {critique}

Generate an improved version that addresses the critique.
Return only the code.""",
            system="You are an expert code optimiser. Improve the solution."
        )
        return Candidate(
            code=mutated_code,
            generation=self.generation,
            parent_id=parent.candidate_id
        )

    def run(self, generations: int = 3) -> Candidate:
        print(f"\n🧬 EVOLUTIONARY OPTIMISATION")
        print(f"   Task: {self.task}")
        print(f"   Population: {self.population_size} | Generations: {generations}")

        print("\n[Gen 0] Generating initial population...")
        self.population = self._generate_initial()

        for gen in range(1, generations + 1):
            self.generation = gen
            print(f"\n[Gen {gen}] Evaluating {len(self.population)} candidates...")

            critiques = []
            for c in self.population:
                score, critique = self.evaluator(self.task, c.code)
                c.score         = score
                critiques.append(critique)
                print(f"  Candidate {c.candidate_id}: score={score:.2f}")

            self.population.sort(key=lambda c: c.score, reverse=True)
            best = self.population[0]

            if self.best_ever is None or best.score > self.best_ever.score:
                self.best_ever = copy.deepcopy(best)
                print(f"  🏆 New best: {best.candidate_id} (score={best.score:.2f})")

            self.history.append({
                "generation": gen,
                "best_score": best.score,
                "avg_score":  sum(c.score for c in self.population) / len(self.population)
            })

            if gen < generations:
                print(f"  → Mutating top candidates...")
                survivors  = self.population[:max(1, self.population_size // 2)]
                new_pop    = list(survivors)
                for parent, critique in zip(survivors, critiques):
                    if len(new_pop) < self.population_size:
                        new_pop.append(self._mutate(parent, critique))
                self.population = new_pop

        return self.best_ever

    def plot_progress(self):
        print("\n📊 Evolutionary Progress:")
        for h in self.history:
            bar = "█" * int(h["best_score"] * 20)
            print(f"  Gen {h['generation']}: [{bar:<20}] best={h['best_score']:.2f} avg={h['avg_score']:.2f}")

def code_evaluator(task: str, code: str) -> Tuple[float, str]:
    'Evaluate code quality using LLM as judge.'
    raw = call_llm(
        f"""Evaluate this Python code for: {task}
Criteria: correctness, efficiency, readability, edge case handling.
Return ONLY JSON: {{"score": <0.0-1.0>, "critique": "<issues>"}}

Code:
{code}""",
        system="You are a strict code evaluator. Return only valid JSON."
    )
    try:
        data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
        return float(data.get("score", 0.5)), data.get("critique", "")
    except:
        return 0.5, "Evaluation failed"

evolve = EvolutionaryOptimiser(
    task="Write a function that finds all prime numbers up to n using the Sieve of Eratosthenes",
    evaluator_fn=code_evaluator,
    population_size=3
)

best = evolve.run(generations=2)
evolve.plot_progress()

print(f"\n{'='*60}\nBEST SOLUTION (score={best.score:.2f})\n{'='*60}")
print(best.code)

## 10. Supervised Learning — LLM as Label Generator

Use an LLM to generate labelled training data for a downstream ML model. The ML model then handles inference cheaply at scale.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

print("[Step 1] LLM generating labelled training data...")

raw = call_llm(
    """Generate 20 customer support tickets with their urgency labels.
Return ONLY a JSON array. Each item: {"text": "...", "label": "URGENT"|"NORMAL"|"LOW"}
Make sure to include at least 6 of each label.""",
    system="You are a training data generator. Return only valid JSON.",
    max_tokens=2048
)
training_data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
print(f"✅ Generated {len(training_data)} labelled examples")

from collections import Counter
dist = Counter(d["label"] for d in training_data)
print(f"   Distribution: {dict(dist)}")

print("\n[Step 2] Training ML classifier on LLM-generated data...")

texts  = [d["text"]  for d in training_data]
labels = [d["label"] for d in training_data]

if set(labels) != {"URGENT", "NORMAL", "LOW"} or min(Counter(labels).values()) < 2:
    raise ValueError("Generate at least two examples for each of URGENT, NORMAL, and LOW, then rerun this cell.")
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.4, random_state=42, stratify=labels
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf",   LogisticRegression(max_iter=1000, C=5))
])
pipeline.fit(X_train, y_train)

print("\n[Step 3] Evaluating classifier...")
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, labels=["URGENT", "NORMAL", "LOW"], target_names=["URGENT", "NORMAL", "LOW"], zero_division=0))

print("[Step 4] Fast inference (no LLM cost):")
test_tickets = [
    "Production database is completely down!",
    "Can we add a night mode feature?",
    "Password reset link expired.",
]
for ticket in test_tickets:
    pred  = pipeline.predict([ticket])[0]
    proba = pipeline.predict_proba([ticket])[0] if hasattr(pipeline, 'predict_proba') else None
    print(f"  [{pred}] {ticket}")

## 11. SICA Architecture Summary

In [ ]:
sica_architecture = """
╔══════════════════════════════════════════════════════════════╗
║              SICA SELF-IMPROVEMENT ARCHITECTURE              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║   ┌─────────────┐    ┌──────────────┐    ┌──────────────┐  ║
║   │   ARCHIVE   │    │   EXECUTOR   │    │   EVALUATOR  │  ║
║   │  (versions) │───▶│  (best ver)  │───▶│  (benchmarks)│  ║
║   │  v1, v2...  │    │              │    │              │  ║
║   └─────────────┘    └──────────────┘    └──────┬───────┘  ║
║         ▲                                        │          ║
║         │                                        ▼          ║
║   ┌─────┴──────────────────────────────────────────────┐   ║
║   │              SELF-MODIFIER (LLM)                    │   ║
║   │  "Analyse archive + critique → rewrite instructions"│   ║
║   └─────────────────────────────────────────────────────┘   ║
║         ▲                                                    ║
║         │  (monitors for stagnation / loops)                 ║
║   ┌─────┴──────────┐                                        ║
║   │   OVERSEER     │ ← async monitor, can halt execution    ║
║   └────────────────┘                                        ║
╠══════════════════════════════════════════════════════════════╣
║  Key SICA Learnings (from paper):                            ║
║  • Started with basic file overwrite                         ║
║  • Self-developed: Smart Editor → Diff Editor → AST Locator  ║
║  • Docker containerisation critical for safety               ║
║  • Overseer prevents infinite loops and stagnation           ║
║  • Sub-agents manage context length for long iterations      ║
╚══════════════════════════════════════════════════════════════╝
"""
print(sica_architecture)

## 12. Learning Pattern Selection Guide

In [ ]:
guide = """
╔════════════════════════╦═════════════════════════╦════════════════════════════╗
║ Pattern                ║ Use When                ║ Key Mechanism              ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Few-Shot               ║ New task, minimal data  ║ Examples in prompt         ║
║ (In-Context)           ║ Rapid adaptation needed ║ No training required       ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Reward-Driven (RL)     ║ Sequential decisions    ║ Reward → policy update     ║
║                        ║ Game / control agents   ║ PPO clipping for stability ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Online Learning        ║ Streaming data          ║ Incremental model update   ║
║                        ║ Concept drift expected  ║ SGD / partial_fit          ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Memory-Based           ║ Recurring problem types ║ Episodic recall + retrieval║
║                        ║ Personalised agents     ║ Filter by success/rating   ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Self-Improvement       ║ Long-running autonomous ║ Execute → Evaluate →       ║
║ (SICA)                 ║ coding / research tasks ║ Self-modify → Archive      ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Evolutionary           ║ Algorithm optimisation  ║ Population → Select →      ║
║ (AlphaEvolve)          ║ No known solution path  ║ LLM Mutate → Evaluate      ║
╠════════════════════════╬═════════════════════════╬════════════════════════════╣
║ Supervised (LLM-Gen)   ║ Build cheap classifier  ║ LLM labels → train sklearn ║
║                        ║ High-volume inference   ║ Fast prediction, no LLM    ║
╚════════════════════════╩═════════════════════════╩════════════════════════════╝

⚠️  SAFETY NOTE (from SICA paper):
   Self-improving agents that execute code MUST run in Docker containers.
   Always use an overseer to detect loops, stagnation, or unsafe actions.
"""
print(guide)

## 13. Exercises

- **Exercise 1 — Reward Shaping:** Modify `RLAgent` to distinguish between immediate rewards (is the response good?) and delayed rewards (did the issue get resolved?). Track both separately and weight them 70/30 in policy updates.

- **Exercise 2 — Population Diversity:** Extend `EvolutionaryOptimiser` to maintain diversity: if two candidates have more than 80% overlap (use simple word overlap), force a mutation of the lower-scoring one before selection.

- **Exercise 3 — DPO Dataset Generator:** Build a tool that generates DPO training pairs: given a prompt, the LLM generates two responses, an evaluator labels them as `chosen` and `rejected`, and the pairs are saved as a JSONL file suitable for fine-tuning.

- **Exercise 4 — SICA with Unit Tests:** Extend `SelfImprovingAgent` to evaluate code by actually running it against a suite of unit tests (use `subprocess.run` in a sandboxed way). Score = fraction of tests passed.

- **Exercise 5 — Adaptive Persona:** Build an agent that tracks which communication style (formal, casual, technical, simple) achieves the highest user satisfaction scores. After 10 interactions, the agent automatically switches to the best-performing style.

**Next:** Chapter 10 — Model Context Protocol (MCP)